# Lab 1: Embeddings Visualization

**Level:** Basic | **Duration:** ~30 minutes

## What You'll Learn
- How text embeddings represent meaning as vectors
- How to generate embeddings using OpenAI's `text-embedding-3-small` model
- How to measure semantic similarity with cosine similarity
- How to visualize high-dimensional embeddings in 2D using t-SNE
- How semantically similar sentences cluster together in embedding space

## Why This Matters
Embeddings are the foundation of every RAG pipeline. Understanding how they capture meaning is critical before you build retrieval systems.

## Setup
Install dependencies and configure your API key.

In [ ]:
!pip install -q openai numpy matplotlib scikit-learn

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "your-key-here"

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI

client = OpenAI()
print("Setup complete!")

## Step 1: Define Sentences by Category

We'll create sentences from three distinct categories. If embeddings work well, sentences from the same category should end up close together in vector space.

In [ ]:
sentences = {
    "Sports": [
        "The goalkeeper made an incredible save in the final minute.",
        "She sprinted past the defender and scored a hat-trick.",
        "The tennis match went to five sets before the champion won.",
        "He broke the world record in the 100-meter dash.",
        "The basketball team won the championship in overtime.",
    ],
    "Technology": [
        "The new processor delivers 40% better performance per watt.",
        "Kubernetes orchestrates containerized applications at scale.",
        "The latest smartphone features a 200-megapixel camera sensor.",
        "Machine learning models are getting smaller and faster every year.",
        "The database query was optimized to run in under 10 milliseconds.",
    ],
    "Cooking": [
        "Simmer the sauce on low heat for at least thirty minutes.",
        "Fresh basil and mozzarella make the perfect caprese salad.",
        "Knead the dough until it becomes smooth and elastic.",
        "Caramelize the onions slowly to bring out their natural sweetness.",
        "The sourdough starter needs to be fed every twelve hours.",
    ],
}

# Flatten for embedding
all_sentences = []
labels = []
for category, sents in sentences.items():
    for s in sents:
        all_sentences.append(s)
        labels.append(category)

print(f"Total sentences: {len(all_sentences)}")
for cat, sents in sentences.items():
    print(f"  {cat}: {len(sents)} sentences")

## Step 2: Generate Embeddings

We'll use OpenAI's `text-embedding-3-small` model. Each sentence becomes a vector of 1536 dimensions.

In [ ]:
def get_embeddings(texts, model="text-embedding-3-small"):
    """Get embeddings for a list of texts using OpenAI API."""
    response = client.embeddings.create(input=texts, model=model)
    return [item.embedding for item in response.data]

embeddings = get_embeddings(all_sentences)
embeddings_array = np.array(embeddings)

print(f"Embedding shape: {embeddings_array.shape}")
print(f"Each sentence is represented by a {embeddings_array.shape[1]}-dimensional vector.")

## Step 3: Cosine Similarity

Cosine similarity measures how similar two vectors are, regardless of their magnitude. A value of 1.0 means identical direction, 0.0 means orthogonal (unrelated), and -1.0 means opposite.

Let's compare a few pairs to build intuition.

In [ ]:
# Compare specific pairs
pairs = [
    (0, 1),   # Sports vs Sports
    (0, 5),   # Sports vs Technology
    (0, 10),  # Sports vs Cooking
    (5, 6),   # Technology vs Technology
    (10, 11), # Cooking vs Cooking
]

print("Cosine Similarity Between Sentence Pairs")
print("=" * 70)
for i, j in pairs:
    sim = cosine_similarity([embeddings[i]], [embeddings[j]])[0][0]
    print(f"\n[{labels[i]}] \"{all_sentences[i][:50]}...\"")
    print(f"[{labels[j]}] \"{all_sentences[j][:50]}...\"")
    print(f"Similarity: {sim:.4f}")

## Step 4: Full Similarity Matrix

Let's visualize the full pairwise similarity matrix as a heatmap. You should see brighter blocks along the diagonal where same-category sentences are compared.

In [ ]:
sim_matrix = cosine_similarity(embeddings_array)

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(sim_matrix, cmap="YlOrRd", vmin=0, vmax=1)

# Add category labels
ax.set_xticks(range(len(all_sentences)))
ax.set_yticks(range(len(all_sentences)))
ax.set_xticklabels([f"{l[0]}{i}" for i, l in enumerate(labels)], rotation=90, fontsize=8)
ax.set_yticklabels([f"{l[0]}{i}" for i, l in enumerate(labels)], fontsize=8)

# Add category separators
for pos in [5, 10]:
    ax.axhline(y=pos - 0.5, color="black", linewidth=2)
    ax.axvline(x=pos - 0.5, color="black", linewidth=2)

plt.colorbar(im, label="Cosine Similarity")
plt.title("Cosine Similarity Matrix (S=Sports, T=Technology, C=Cooking)")
plt.tight_layout()
plt.show()

**Observation:** Notice the 3 bright blocks along the diagonal? Those are within-category similarities. Cross-category similarities are noticeably lower. This is the core insight: embeddings capture semantic meaning.

## Step 5: Visualize with t-SNE

t-SNE (t-distributed Stochastic Neighbor Embedding) reduces high-dimensional vectors to 2D while preserving local structure. Similar points stay close together.

In [ ]:
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(embeddings_array)

colors = {"Sports": "#e74c3c", "Technology": "#3498db", "Cooking": "#2ecc71"}

fig, ax = plt.subplots(figsize=(12, 8))

for category in sentences.keys():
    mask = [l == category for l in labels]
    points = embeddings_2d[mask]
    ax.scatter(points[:, 0], points[:, 1],
               c=colors[category], label=category,
               s=100, alpha=0.8, edgecolors="white", linewidth=1.5)

# Annotate each point with a short version of the sentence
for i, (x, y) in enumerate(embeddings_2d):
    short = all_sentences[i][:35] + "..."
    ax.annotate(short, (x, y), fontsize=7, alpha=0.7,
                xytext=(5, 5), textcoords="offset points")

ax.legend(fontsize=12)
ax.set_title("t-SNE Visualization of Sentence Embeddings", fontsize=14)
ax.set_xlabel("t-SNE dimension 1")
ax.set_ylabel("t-SNE dimension 2")
plt.tight_layout()
plt.show()

## Step 6: Quantify Cluster Quality

Let's measure the average within-category vs. between-category similarity to put numbers on what we see.

In [ ]:
within_sims = []
between_sims = []

for i in range(len(all_sentences)):
    for j in range(i + 1, len(all_sentences)):
        sim = sim_matrix[i][j]
        if labels[i] == labels[j]:
            within_sims.append(sim)
        else:
            between_sims.append(sim)

print(f"Average within-category similarity:  {np.mean(within_sims):.4f}")
print(f"Average between-category similarity: {np.mean(between_sims):.4f}")
print(f"Gap: {np.mean(within_sims) - np.mean(between_sims):.4f}")
print(f"\nA larger gap means better semantic separation.")

## Step 7: Find the Most Similar Sentence

Given a query, find the most similar sentence in our collection. This is essentially what vector search does.

In [ ]:
query = "The athlete finished the race in record time."
query_embedding = get_embeddings([query])[0]

similarities = cosine_similarity([query_embedding], embeddings_array)[0]

# Rank by similarity
ranked_indices = np.argsort(similarities)[::-1]

print(f"Query: \"{query}\"\n")
print("Top 5 most similar sentences:")
print("-" * 60)
for rank, idx in enumerate(ranked_indices[:5], 1):
    print(f"{rank}. [{labels[idx]}] {all_sentences[idx]}")
    print(f"   Similarity: {similarities[idx]:.4f}\n")

## Step 8: Embedding Dimensions Matter

Let's compare `text-embedding-3-small` (1536 dims) with `text-embedding-3-large` (3072 dims) on the same data. Does the larger model produce better separation?

In [ ]:
# Get embeddings from the larger model
embeddings_large = get_embeddings(all_sentences, model="text-embedding-3-large")
embeddings_large_array = np.array(embeddings_large)

sim_matrix_large = cosine_similarity(embeddings_large_array)

within_large = []
between_large = []
for i in range(len(all_sentences)):
    for j in range(i + 1, len(all_sentences)):
        sim = sim_matrix_large[i][j]
        if labels[i] == labels[j]:
            within_large.append(sim)
        else:
            between_large.append(sim)

print("Model Comparison")
print("=" * 50)
print(f"{'Metric':<35} {'Small':>7} {'Large':>7}")
print("-" * 50)
print(f"{'Dimensions':<35} {'1536':>7} {'3072':>7}")
print(f"{'Avg within-category similarity':<35} {np.mean(within_sims):>7.4f} {np.mean(within_large):>7.4f}")
print(f"{'Avg between-category similarity':<35} {np.mean(between_sims):>7.4f} {np.mean(between_large):>7.4f}")
print(f"{'Gap (higher = better separation)':<35} {np.mean(within_sims)-np.mean(between_sims):>7.4f} {np.mean(within_large)-np.mean(between_large):>7.4f}")

---

## YOUR TURN

Add your own category of 5 sentences below. Ideas:
- **Travel**: "The flight from Istanbul to London takes about 4 hours."
- **Medicine**: "The patient was prescribed antibiotics for the infection."
- **Finance**: "The stock market crashed after the interest rate hike."

Then re-run the t-SNE visualization. Does your new category form its own cluster?

In [ ]:
# YOUR TURN: Add a new category
my_sentences = [
    # Add 5 sentences from a new category here
    "Your sentence 1",
    "Your sentence 2",
    "Your sentence 3",
    "Your sentence 4",
    "Your sentence 5",
]
my_category = "MyCategory"  # Name your category

# Combine with existing sentences
combined_sentences = all_sentences + my_sentences
combined_labels = labels + [my_category] * len(my_sentences)

# Get embeddings for all
combined_embeddings = get_embeddings(combined_sentences)
combined_array = np.array(combined_embeddings)

# t-SNE visualization
tsne_combined = TSNE(n_components=2, random_state=42, perplexity=5)
combined_2d = tsne_combined.fit_transform(combined_array)

colors_extended = {**colors, my_category: "#9b59b6"}

fig, ax = plt.subplots(figsize=(12, 8))
for category in set(combined_labels):
    mask = [l == category for l in combined_labels]
    points = combined_2d[mask]
    ax.scatter(points[:, 0], points[:, 1],
               c=colors_extended.get(category, "#95a5a6"),
               label=category, s=100, alpha=0.8,
               edgecolors="white", linewidth=1.5)

ax.legend(fontsize=12)
ax.set_title("t-SNE with Your Custom Category", fontsize=14)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **Embeddings capture meaning** — similar sentences have similar vectors, regardless of word overlap.
2. **Cosine similarity** is the standard way to measure closeness in embedding space.
3. **t-SNE** lets you visualize high-dimensional structure in 2D — useful for debugging and intuition.
4. **Model choice matters** — larger embedding models may produce better separation at higher cost.
5. This same principle powers vector search in RAG: embed a query, find the closest document chunks.

**Next:** In Lab 2, we'll use these embeddings to build actual search with ChromaDB.